# InstaSHAP Replication — Clean Implementation (Google Colab Ready)

**Paper:** InstaSHAP: Interpretable Additive Models Explain Shapley Values Instantly (ICLR 2025)

**Paper Link:** [https://openreview.net/forum?id=ky7vVlBQBY](https://openreview.net/forum?id=ky7vVlBQBY)

**Author:** J Ganesh Reddy && Team

---

## Datasets Used
As specified, we strictly use the exact datasets from the paper:
1. **Bike Sharing** (UCI) — Syndertistic interactions (hour × workday)
2. **Forest CoverType** (sklearn) — Redundant interactions (classification)
3. **Adult Income** (UCI) — Supplementary tabular results
4. **Synthetic** — 10-dim Gaussian, 4 settings (k*∈{1,2}, ρ∈{0,0.707})

*(Dataset 4: CUB Birds from Section 7 is excluded due to large ResNet fine-tuning requirements.)*

## What We Replicate
- **Part 1:** Accuracy comparison — GAM-1 vs Low-Dim GAM vs MLP for all 3 tabular datasets
- **Part 2:** Figure 4 — Hour × Workday interaction heatmap (flagship visualization)
- **Part 3:** Figure 3 — InstaSHAP vs FastSHAP MSE convergence (synthetic experiment)

## Cell 0: Colab Setup (Install Dependencies)
Run this cell first if you are running on Google Colab.

In [ ]:
!pip install shap interpret xgboost scikit-learn matplotlib seaborn pandas numpy

## Cell 1: Imports & Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
import time
import os
import urllib.request
import zipfile
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_absolute_error, accuracy_score
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder
from interpret.glassbox import ExplainableBoostingRegressor, ExplainableBoostingClassifier

np.random.seed(42)

# Create directories for saving outputs
os.makedirs('results/figures', exist_ok=True)

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print('✓ All imports successful! Ready to run experiments.')

## Cell 2: Download & Load Bike Sharing Dataset

In [ ]:
# Use the explicit physical paths provided by the user
bike_csv = r'D:\X-AI\X_AI-Project\data\hour.csv'
day_csv  = r'D:\X-AI\X_AI-Project\data\day.csv'

if not os.path.exists(bike_csv):
    raise FileNotFoundError(f"Dataset not found at physical path: {bike_csv}")

df_bike = pd.read_csv(bike_csv)

# Features exactly as used in the paper (13 features)
BIKE_FEATURES = ['season', 'yr', 'mnth', 'hr', 'holiday', 'weekday',
                 'workingday', 'weathersit', 'temp', 'atemp', 'hum',
                 'windspeed', 'casual']
BIKE_TARGET = 'cnt'  # total bike count (regression target)

X_bike = df_bike[BIKE_FEATURES].values
y_bike = df_bike[BIKE_TARGET].values

X_bike_train, X_bike_test, y_bike_train, y_bike_test = train_test_split(
    X_bike, y_bike, test_size=0.2, random_state=42
)

print(f'\nBike Sharing Dataset loaded from {bike_csv}:')
print(f'  Train: {X_bike_train.shape}, Test: {X_bike_test.shape}')

## Cell 3: Load Covertype Dataset

In [ ]:
from sklearn.datasets import fetch_covtype

print('Loading Covertype dataset from sklearn...')
data_cov = fetch_covtype()
X_cov = data_cov.data        # 54 features (10 numeric + 44 binary)
y_cov = data_cov.target - 1  # 7 classes, 0-indexed

# Paper uses first 10 numeric features only
X_cov_numeric = X_cov[:, :10]

X_cov_train, X_cov_test, y_cov_train, y_cov_test = train_test_split(
    X_cov_numeric, y_cov, test_size=0.2, random_state=42, stratify=y_cov
)

# --- DEV MODE: SUBSAMPLE --- 
idx = np.random.choice(len(X_cov_train), 200000, replace=False)
X_cov_train = X_cov_train[idx]
y_cov_train = y_cov_train[idx]

print(f'\u2713 Covertype Dataset loaded (Dev Mode Subsampled):')
print(f'  Train: {X_cov_train.shape}, Test: {X_cov_test.shape} (10 features)')

## Cell 4: Load Adult Income Dataset

In [ ]:
print("Downloading/Loading Adult Income dataset...")
url_adult = "https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data"
columns = ["age", "workclass", "fnlwgt", "education", "education-num", "marital-status",
           "occupation", "relationship", "race", "sex", "capital-gain", "capital-loss",
           "hours-per-week", "native-country", "income"]
df_adult = pd.read_csv(url_adult, names=columns, sep=r',\s*', engine='python')

# Drop rows with missing values indicated by '?'
df_adult = df_adult.replace('?', np.nan).dropna()

X_adult_raw = df_adult.drop('income', axis=1)
y_adult = (df_adult['income'] == '>50K').astype(int).values

# Encode categorical variables so MLP can process them
enc = OrdinalEncoder()
cat_cols = X_adult_raw.select_dtypes(include=['object']).columns
X_adult = X_adult_raw.copy()
X_adult[cat_cols] = enc.fit_transform(X_adult_raw[cat_cols])
X_adult = X_adult.values

X_adult_train, X_adult_test, y_adult_train, y_adult_test = train_test_split(
    X_adult, y_adult, test_size=0.2, random_state=42, stratify=y_adult
)

print(f'\u2713 Adult Income Dataset loaded:')
print(f'  Train: {X_adult_train.shape}, Test: {X_adult_test.shape}')

---
## Part 1: Accuracy Comparison — GAM vs MLP
The core claim: Low-dimensional GAM matches black-box MLP accuracy.

In [ ]:
### Cell 5: Bike Sharing (Regression) ###
scaler_bike = StandardScaler()
X_bike_train_sc = scaler_bike.fit_transform(X_bike_train)
X_bike_test_sc  = scaler_bike.transform(X_bike_test)

print('Training Bike Sharing Models...')

# 1. MLP (Dev Mode parameters)
mlp_bike = MLPRegressor(hidden_layer_sizes=(128, 64, 32), activation='relu', max_iter=150, early_stopping=True, n_iter_no_change=10, random_state=42)
mlp_bike.fit(X_bike_train_sc, y_bike_train)
bike_mlp_err = 1 - r2_score(y_bike_test, mlp_bike.predict(X_bike_test_sc))
print(f'  \u2713 MLP Error:           {bike_mlp_err*100:.2f}%  (paper: ~6.59%)')

# 2. GAM-1
ebm_bike_gam1 = ExplainableBoostingRegressor(interactions=0, max_bins=256, n_jobs=-1, random_state=42)
ebm_bike_gam1.fit(X_bike_train, y_bike_train)
bike_gam1_err = 1 - r2_score(y_bike_test, ebm_bike_gam1.predict(X_bike_test))
print(f'  \u2713 GAM-1 Error:         {bike_gam1_err*100:.2f}%  (paper: ~17.4%)')

# 3. Low-Dim GAM (Dev Mode interactions=5)
ebm_bike_gam2 = ExplainableBoostingRegressor(interactions=5, max_bins=256, n_jobs=-1, random_state=42)
ebm_bike_gam2.fit(X_bike_train, y_bike_train)
bike_gam2_err = 1 - r2_score(y_bike_test, ebm_bike_gam2.predict(X_bike_test))
print(f'  \u2713 Low-Dim GAM Error:   {bike_gam2_err*100:.2f}%  (paper: ~6.23%)')

In [ ]:
### Cell 6: Covertype (Classification) ###
scaler_cov = StandardScaler()
X_cov_train_sc = scaler_cov.fit_transform(X_cov_train)
X_cov_test_sc  = scaler_cov.transform(X_cov_test)

print('Training Covertype Models (May take 10-20 minutes)...')

# 1. MLP (Dev Mode parameters)
mlp_cov = MLPClassifier(hidden_layer_sizes=(128, 64), max_iter=150, early_stopping=True, n_iter_no_change=10, random_state=42)
mlp_cov.fit(X_cov_train_sc, y_cov_train)
cov_mlp_acc = accuracy_score(y_cov_test, mlp_cov.predict(X_cov_test_sc))
print(f'  \u2713 MLP Accuracy:         {cov_mlp_acc*100:.1f}%  (paper: ~80.4%)')

# 2. GAM-1
ebm_cov_gam1 = ExplainableBoostingClassifier(interactions=0, max_bins=256, n_jobs=-1, random_state=42)
ebm_cov_gam1.fit(X_cov_train, y_cov_train)
cov_gam1_acc = accuracy_score(y_cov_test, ebm_cov_gam1.predict(X_cov_test))
print(f'  \u2713 GAM-1 Accuracy:       {cov_gam1_acc*100:.1f}%  (paper: ~72.4%)')

# 3. Low-Dim GAM (Dev Mode interactions=5)
ebm_cov_gam2 = ExplainableBoostingClassifier(interactions=5, max_bins=256, n_jobs=-1, random_state=42)
ebm_cov_gam2.fit(X_cov_train, y_cov_train)
cov_gam2_acc = accuracy_score(y_cov_test, ebm_cov_gam2.predict(X_cov_test))
print(f'  \u2713 Low-Dim GAM Accuracy: {cov_gam2_acc*100:.1f}%  (paper: ~82.2%)')

In [ ]:
### Cell 7: Adult Income (Classification) ###
scaler_adult = StandardScaler()
X_adult_train_sc = scaler_adult.fit_transform(X_adult_train)
X_adult_test_sc  = scaler_adult.transform(X_adult_test)

print('Training Adult Income Models...')

# 1. MLP (Dev Mode parameters)
mlp_adult = MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=150, early_stopping=True, n_iter_no_change=10, random_state=42)
mlp_adult.fit(X_adult_train_sc, y_adult_train)
adult_mlp_acc = accuracy_score(y_adult_test, mlp_adult.predict(X_adult_test_sc))
print(f'  \u2713 MLP Accuracy:         {adult_mlp_acc*100:.2f}%')

# 2. GAM-1
ebm_adult_gam1 = ExplainableBoostingClassifier(interactions=0, max_bins=256, n_jobs=-1, random_state=42)
ebm_adult_gam1.fit(X_adult_train, y_adult_train)
adult_gam1_acc = accuracy_score(y_adult_test, ebm_adult_gam1.predict(X_adult_test))
print(f'  \u2713 GAM-1 Accuracy:       {adult_gam1_acc*100:.2f}%')

# 3. Low-Dim GAM (Dev Mode interactions=5)
ebm_adult_gam2 = ExplainableBoostingClassifier(interactions=5, max_bins=256, n_jobs=-1, random_state=42)
ebm_adult_gam2.fit(X_adult_train, y_adult_train)
adult_gam2_acc = accuracy_score(y_adult_test, ebm_adult_gam2.predict(X_adult_test))
print(f'  \u2713 Low-Dim GAM Accuracy: {adult_gam2_acc*100:.2f}%')

In [ ]:
### Cell 8: Main Results Table ###
print('\n' + '='*85)
print('INSTASHAP REPLICATION \u2014 MULTI-DATASET RESULTS')
print('='*85)
print(f"{'Model':<30} {'Bike Error ↓':>14} {'Covertype Acc ↑':>18} {'Adult Acc ↑':>16}")
print('-'*85)
print(f"{'Paper: MLP Black-Box':<30} {'~6.59%':>14} {'~80.4%':>18} {'-':>16}")
print(f"{'Paper: GAM-1':<30} {'~17.4%':>14} {'~72.4%':>18} {'-':>16}")
print(f"{'Paper: Low-Dim GAM':<30} {'~6.23%':>14} {'~82.2%':>18} {'-':>16}")
print('-'*85)
print(f"{'Ours: MLP Black-Box':<30} {bike_mlp_err*100:>13.2f}% {cov_mlp_acc*100:>17.1f}% {adult_mlp_acc*100:>15.2f}%")
print(f"{'Ours: GAM-1':<30} {bike_gam1_err*100:>13.2f}% {cov_gam1_acc*100:>17.1f}% {adult_gam1_acc*100:>15.2f}%")
print(f"{'Ours: Low-Dim GAM':<30} {bike_gam2_err*100:>13.2f}% {cov_gam2_acc*100:>17.1f}% {adult_gam2_acc*100:>15.2f}%")
print('='*85)

# Create matplotlib table
fig, ax = plt.subplots(figsize=(10, 5))
ax.axis('off')

table_data = [
    ['', 'Bike Error ↓', 'Covertype Acc ↑', 'Adult Acc ↑'],
    ['Paper: MLP Black-Box', '~6.59%', '~80.4%', '-'],
    ['Paper: GAM-1', '~17.4%', '~72.4%', '-'],
    ['Paper: Low-Dim GAM', '~6.23%', '~82.2%', '-'],
    ['Ours: MLP Black-Box', f'{bike_mlp_err*100:.2f}%', f'{cov_mlp_acc*100:.1f}%', f'{adult_mlp_acc*100:.2f}%'],
    ['Ours: GAM-1', f'{bike_gam1_err*100:.2f}%', f'{cov_gam1_acc*100:.1f}%', f'{adult_gam1_acc*100:.2f}%'],
    ['Ours: Low-Dim GAM', f'{bike_gam2_err*100:.2f}%', f'{cov_gam2_acc*100:.1f}%', f'{adult_gam2_acc*100:.2f}%'],
]

table = ax.table(cellText=table_data, loc='center', cellLoc='center')
table.auto_set_font_size(False)
table.set_fontsize(11)
table.scale(1.2, 1.8)
for j in range(4):
    table[0, j].set_facecolor('#4472C4')
    table[0, j].set_text_props(color='white', fontweight='bold')
for i in range(1, 4):
    for j in range(4): table[i, j].set_facecolor('#D9E2F3')
for i in range(4, 7):
    for j in range(4): table[i, j].set_facecolor('#E2EFDA')

ax.set_title('InstaSHAP Replication: Combined Dataset Results', fontsize=14, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('results/figures/results_table.png', dpi=150, bbox_inches='tight')
plt.show()

---
## Part 2: Figure 4 Reproduction — Hour × Workday Interaction
This reproduces the paper's flagship visualization.

In [ ]:
### Cell 9: Figure 4 Heatmap ###
hr_idx  = BIKE_FEATURES.index('hr')
wd_idx  = BIKE_FEATURES.index('workingday')
hrs      = np.arange(0, 24)
workdays = [0, 1]

grid = np.zeros((2, 24))
for w in workdays:
    for h in hrs:
        sample = np.median(X_bike_train, axis=0).copy()
        sample[hr_idx]  = h
        sample[wd_idx]  = w
        grid[w, h] = ebm_bike_gam2.predict(sample.reshape(1, -1))[0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
ax = axes[0]
ax.plot(hrs, grid[0], label='Weekend/Holiday', marker='o', color='#E74C3C', linewidth=2)
ax.plot(hrs, grid[1], label='Workday', marker='s', color='#3498DB', linewidth=2)
ax.set_xlabel('Hour of Day', fontsize=12)
ax.set_ylabel('Predicted Bike Demand', fontsize=12)
ax.set_title('Figure 4a: GAM Hour × Workday Interaction', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

ax = axes[1]
heatmap_data = pd.DataFrame(grid, index=['Weekend', 'Workday'], columns=hrs)
sns.heatmap(heatmap_data, ax=ax, cmap='YlOrRd', cbar_kws={'label': 'Predicted Demand'})
ax.set_title('Figure 4b: Heatmap', fontsize=13)

plt.tight_layout()
plt.savefig('results/figures/figure4_reproduction.png', dpi=150)
plt.show()

---
## Part 3: Figure 3 Reproduction — InstaSHAP vs FastSHAP (Synthetic)
⚠️ **Note:** This cell takes 15-20 minutes.

In [ ]:
### Cell 10: Figure 3 ###
def generate_synthetic(n=5000, d=10, k_star=1, rho=0.0, seed=42):
    rng = np.random.RandomState(seed)
    cov = np.full((d, d), rho)
    np.fill_diagonal(cov, 1.0)
    X = rng.multivariate_normal(np.zeros(d), cov, size=n)
    if k_star == 1:
        y = X[:, 0] * 2 + X[:, 1] ** 2 + X[:, 2]
    else:
        y = X[:, 0] * X[:, 1] + X[:, 2] * X[:, 3] + X[:, 4]
    return X, y

def compute_shap_labels(X, y):
    from sklearn.linear_model import Ridge
    model = Ridge().fit(X, y)
    # DEV MODE SHAP SIZE: bg=50, instead of 200. target=200 instead of 500.
    bg = shap.sample(pd.DataFrame(X), 50)
    explainer = shap.KernelExplainer(model.predict, bg)
    return explainer.shap_values(X[:200])

settings = [(1, 0.0), (1, 0.707), (2, 0.0), (2, 0.707)]
titles   = ['k*=1, ρ=0.0', 'k*=1, ρ=0.707', 'k*=2, ρ=0.0', 'k*=2, ρ=0.707']
fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for idx, ((k_star, rho), title) in enumerate(zip(settings, titles)):
    print(f'Running: {title}...')
    X_syn, y_syn = generate_synthetic(k_star=k_star, rho=rho)
    shap_gt = compute_shap_labels(X_syn, y_syn)
    
    # DEV MODE SYNTHETIC: epochs=15 instead of 30
    epochs, instashap_mse, fastshap_mse = 15, [], []
    for ep in range(1, epochs + 1):
        ebm = ExplainableBoostingRegressor(interactions=0, max_bins=64, max_rounds=ep*5, n_jobs=-1, random_state=42)
        ebm.fit(X_syn[:200], shap_gt[:, 0])
        instashap_mse.append(((shap_gt[:, 0] - ebm.predict(X_syn[:200]))**2).mean())
        
        mlp_fs = MLPRegressor(hidden_layer_sizes=(32,), max_iter=ep*3, early_stopping=True, random_state=42)
        mlp_fs.fit(X_syn[:200], shap_gt[:, 0])
        fastshap_mse.append(((shap_gt[:, 0] - mlp_fs.predict(X_syn[:200]))**2).mean())
        
    axes[idx].semilogy(instashap_mse, label='InstaSHAP (EBM)', color='#1D9E75')
    axes[idx].semilogy(fastshap_mse, label='FastSHAP (MLP)', color='#D85A30', linestyle='--')
    axes[idx].set_title(f'SHAP-1 Error ({title})')
    axes[idx].legend()
    axes[idx].grid(True, alpha=0.3)

plt.suptitle('Figure 3 Reproduction: InstaSHAP vs FastSHAP MSE Convergence', fontsize=14, fontweight='bold')
plt.savefig('results/figures/figure3_reproduction.png', dpi=150)
plt.show()